In [1]:
### This code develops random forest based on before and after COVID-19 mandate periods for face mask behaviour binary in Australia.

# Load the pacakges.
import optuna
import pandas as pd
import sklearn.datasets
import sklearn.ensemble
import sklearn.model_selection
import sklearn.metrics
from sklearn.metrics import accuracy_score,recall_score, precision_score, f1_score, roc_auc_score, RocCurveDisplay
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# Set a seed for reproducibility with optuna.
sampler = optuna.samplers.TPESampler(seed=1826179)

# Read the data and select the subset within_mandates == 0.
df1 = pd.read_csv("./face_mask_ausmand.csv")
df = df1[df1['within_mandates'] == 0]
# Drop the following columns to not be added as an explanatory variable.
X = df.drop(columns=["RecordNo",
                     "face_mask_behaviour_scale",
                     "protective_behaviour_scale",
                     "face_mask_behaviour_binary",
                     "protective_behaviour_binary",
                     "protective_behaviour_nomask_scale",
                     "protective_behaviour_nomask_binary",
                     "endtime",
                    "within_mandates"])
# Select the response variable.
y = df['face_mask_behaviour_binary']
# Split the dataset into a training and testing set (0.7, 0.3).
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1826179)

# Define the function for the study which will use the optuna package. Google AI was used to help with this function.
def objective(trial):
    # Select paramaters to be optimised.
    n_estimators = trial.suggest_int("n_estimators", 100, 1000)
    max_depth = trial.suggest_int("max_depth", 1, 20, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 25)
    
    # Create and train model. 
    rf = sklearn.ensemble.RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=1826179
    )
    
    # Cross-Validation based on accuracy results. (Google AI use to help with this step.)
    score = sklearn.model_selection.cross_val_score(
        rf, X_train, y_train, n_jobs=-1, cv=3
    )
    accuracy = score.mean()
    return accuracy

# Create a study to maximise the score value with 20 trials.
study = optuna.create_study(direction="maximize", sampler = sampler)
study.optimize(objective, n_trials=20)

#print(f"Best parameters: {study.best_params}")

# Create the best random forest using best hyperparameter.
best_rf = sklearn.ensemble.RandomForestClassifier(
    **study.best_params, random_state=1826179
)
best_rf.fit(X_train, y_train)
# Calculate the model metrics.
preds = best_rf.predict(X_test)
print(f"Accuracy: {round(sklearn.metrics.accuracy_score(y_test, preds),4)}")
print(f"Recall: {round(sklearn.metrics.recall_score(y_test, preds, pos_label = 'Yes'),4)}")
print(f"Precision: {round(sklearn.metrics.precision_score(y_test, preds, pos_label = 'Yes'),4)}")
print(f"F1 Score: {round(sklearn.metrics.f1_score(y_test, preds, pos_label = 'Yes'),4)}")
probs = best_rf.predict_proba(X_test)[:, 1]

# Convert ['Yes', 'No'] to [1, 0]
y_test_numeric = LabelEncoder().fit_transform(y_test)
auc = roc_auc_score(y_test_numeric, probs)
print(f"ROC AUC: {round(auc,4)}")

[I 2026-05-07 14:10:52,235] A new study created in memory with name: no-name-06e072fb-9901-4788-ab75-973f5df1572f
[I 2026-05-07 14:10:54,272] Trial 0 finished with value: 0.7537550188187243 and parameters: {'n_estimators': 631, 'max_depth': 1, 'min_samples_split': 4}. Best is trial 0 with value: 0.7537550188187243.
[I 2026-05-07 14:10:56,323] Trial 1 finished with value: 0.8023059314020043 and parameters: {'n_estimators': 234, 'max_depth': 20, 'min_samples_split': 4}. Best is trial 1 with value: 0.8023059314020043.
[I 2026-05-07 14:10:59,887] Trial 2 finished with value: 0.7988155802605571 and parameters: {'n_estimators': 707, 'max_depth': 16, 'min_samples_split': 23}. Best is trial 1 with value: 0.8023059314020043.
[I 2026-05-07 14:11:01,089] Trial 3 finished with value: 0.7537550188187243 and parameters: {'n_estimators': 153, 'max_depth': 1, 'min_samples_split': 18}. Best is trial 1 with value: 0.8023059314020043.
[I 2026-05-07 14:11:04,145] Trial 4 finished with value: 0.78019893101

Accuracy: 0.8013
Recall: 0.2935
Precision: 0.7305
F1 Score: 0.4188
ROC AUC: 0.8245


In [2]:
### This code develops classification trees based on before and after COVID-19 mandate periods for face mask behaviour binary in Australia.

# Load the pacakges.
import optuna
import pandas as pd
import sklearn.datasets
import sklearn.ensemble
import sklearn.model_selection
import sklearn.metrics
from sklearn.metrics import accuracy_score,recall_score, precision_score, f1_score, roc_auc_score, RocCurveDisplay
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# Set a seed for reproducibility with optuna.
sampler = optuna.samplers.TPESampler(seed=1826179)

# Read the data and select the subset within_mandates == 1.
df1 = pd.read_csv("./face_mask_ausmand.csv")
df = df1[df1['within_mandates'] == 1]
# Drop the following columns to not be added as an explanatory variable.
X = df.drop(columns=["RecordNo",
                     "face_mask_behaviour_scale",
                     "protective_behaviour_scale",
                     "face_mask_behaviour_binary",
                     "protective_behaviour_binary",
                     "protective_behaviour_nomask_scale",
                     "protective_behaviour_nomask_binary",
                     "endtime",
                    "within_mandates"])
# Select the response variable.
y = df['face_mask_behaviour_binary']
# Split the dataset into a training and testing set (0.7, 0.3).
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1826179)

# Define the function for the study which will use the optuna package. Google AI was used to help with this function.
def objective(trial):
    # Select paramaters to be optimised.
    n_estimators = trial.suggest_int("n_estimators", 100, 1000)
    max_depth = trial.suggest_int("max_depth", 1, 20, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 25)
    
    # Create and train model. 
    rf = sklearn.ensemble.RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=1826179
    )
    
    # Cross-Validation based on accuracy results. (Google AI use to help with this step.)
    score = sklearn.model_selection.cross_val_score(
        rf, X_train, y_train, n_jobs=-1, cv=3
    )
    accuracy = score.mean()
    return accuracy

# Create a study to maximise the score value with 20 trials.
study = optuna.create_study(direction="maximize", sampler = sampler)
study.optimize(objective, n_trials=20)

#print(f"Best parameters: {study.best_params}")

# Create the best random forest using best hyperparameter.
best_rf = sklearn.ensemble.RandomForestClassifier(
    **study.best_params, random_state=1826179
)
best_rf.fit(X_train, y_train)
# Calculate the model metrics.
preds = best_rf.predict(X_test)
print(f"Accuracy: {round(sklearn.metrics.accuracy_score(y_test, preds),4)}")
print(f"Recall: {round(sklearn.metrics.recall_score(y_test, preds, pos_label = 'Yes'),4)}")
print(f"Precision: {round(sklearn.metrics.precision_score(y_test, preds, pos_label = 'Yes'),4)}")
print(f"F1 Score: {round(sklearn.metrics.f1_score(y_test, preds, pos_label = 'Yes'),4)}")
probs = best_rf.predict_proba(X_test)[:, 1]

# Convert ['Yes', 'No'] to [1, 0]
y_test_numeric = LabelEncoder().fit_transform(y_test)
auc = roc_auc_score(y_test_numeric, probs)
print(f"ROC AUC: {round(auc,4)}")

[I 2026-05-07 14:11:31,085] A new study created in memory with name: no-name-4c5f2545-59f0-4c4f-90d1-02488f334ebd
[I 2026-05-07 14:11:32,140] Trial 0 finished with value: 0.7135262004030422 and parameters: {'n_estimators': 631, 'max_depth': 1, 'min_samples_split': 4}. Best is trial 0 with value: 0.7135262004030422.
[I 2026-05-07 14:11:33,965] Trial 1 finished with value: 0.8299384325957614 and parameters: {'n_estimators': 234, 'max_depth': 20, 'min_samples_split': 4}. Best is trial 1 with value: 0.8299384325957614.
[I 2026-05-07 14:11:38,375] Trial 2 finished with value: 0.8204100055799769 and parameters: {'n_estimators': 707, 'max_depth': 16, 'min_samples_split': 23}. Best is trial 1 with value: 0.8299384325957614.
[I 2026-05-07 14:11:38,741] Trial 3 finished with value: 0.7135262004030422 and parameters: {'n_estimators': 153, 'max_depth': 1, 'min_samples_split': 18}. Best is trial 1 with value: 0.8299384325957614.
[I 2026-05-07 14:11:41,970] Trial 4 finished with value: 0.81129592889

Accuracy: 0.8352
Recall: 0.9474
Precision: 0.8415
F1 Score: 0.8913
ROC AUC: 0.8699
